# 02 · The Beam Model & Core Transforms

> **Module 2 · SDK Setup & The Runner · Lectures 2.2–2.4**

In this notebook you will:

- Understand the three abstractions — PCollection, PTransform, PRunner — and how they compose
- Write the same logic four ways: `Map`, `FlatMap`, `Filter`, and `ParDo` with a custom `DoFn`
- Use side inputs to join a small lookup table onto a large stream
- Compare `CombinePerKey` and `GroupByKey`, and know when each one is the right tool
- Use `CoGroupByKey` to join two streams by key
- Prove the associativity and commutativity that every `CombineFn` must satisfy
- Aggregate the course's real order data and cross-check the result against a pure-Python reference

---

## 1. The three abstractions

Everything in Beam is built from three ideas, and once you have them the rest of the API stops looking arbitrary.

| Abstraction | What it is | Everyday example |
|---|---|---|
| **PCollection** | An immutable, distributed collection of elements | "these 5,000 orders" |
| **PTransform** | An operation over one or more PCollections | `beam.Map`, `beam.CombinePerKey` |
| **PRunner** | The thing that executes the graph | `BundleBasedDirectRunner`, `DataflowRunner` |

The pipeline you write is a **graph**, not a sequence. That is the single most important mental shift from writing a script: you declare what the data flow *is*, and the runner decides how to execute it.

```text
PCollection  ──[PTransform]──▶  PCollection
                                    │
                                    ▼
                              executed by a PRunner
```

Because it is a graph and not a script, the runner can **fuse and reorder** your steps. If you write `Map` followed by `Filter`, the runner is free to combine them into a single step that never materialises an intermediate result. That optimisation is not available to you when you write a shell script or a hand-rolled Spark job — it is a direct benefit of expressing the work in the model.

### Two properties of a PCollection that matter

- **Immutable.** A PCollection's contents never change; every transform produces a *new* PCollection. This is why you can consume the same PCollection from five different branches without worrying about ordering or mutation.
- **Lazy.** Creating a PCollection does no work. The graph is built first, and the runner executes it when the pipeline runs.

## 2. Setup for this notebook

We will use the course project for the pure logic and the Beam helpers. The sample data is regenerated only when it is missing, because the generator is seeded and deterministic.

In [1]:
import sys
import subprocess
from pathlib import Path

COURSE_ROOT = Path.cwd()
if not (COURSE_ROOT / "project").exists():
    COURSE_ROOT = Path.cwd().parent

PROJECT_DIR = COURSE_ROOT / "project"
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

if not (COURSE_ROOT / "sample_data" / "orders.csv").exists():
    subprocess.run(
        [sys.executable, str(COURSE_ROOT / "resources" / "sample_data.py"),
         "--output-dir", str(COURSE_ROOT / "sample_data")],
        check=True,
    )

import apache_beam as beam
from apache_beam.options.pipeline_options import PipelineOptions

from transforms import (
    build_revenue_summary,
    format_summary_csv,
    is_valid_order,
    normalise_currency,
    parse_order_line,
    to_summary_row,
    try_parse_order_line,
)

# Use the classic runner explicitly so the output is deterministic and the
# Go-based Prism runner does not take over the pipeline.
LOCAL_OPTIONS = PipelineOptions(["--runner", "BundleBasedDirectRunner"])

print("beam", beam.__version__)
print("course root", COURSE_ROOT)
print("sample data ready:", (COURSE_ROOT / "sample_data" / "orders.csv").exists())


beam 2.77.0
course root /Users/pvishnoi/PycharmProjects/data-etl-ml/data-engineering/data-Course/google/Dataflow
sample data ready: True


## 3. `Map`, `FlatMap`, `Filter` — the three you will use most

These three cover the overwhelming majority of real pipelines.

| Transform | Signature | Cardinality |
|---|---|---|
| `beam.Map` | one element → one element | 1 → 1 |
| `beam.FlatMap` | one element → zero or more elements | 1 → N |
| `beam.Filter` | one element → zero or one element | 1 → 0 or 1 |

The one-to-many case is why `FlatMap` exists. If you use `Map` where you meant `FlatMap`, you get a **nested list** where you wanted a flat stream — and the downstream code then fails in a confusing place, three transforms later.

In [2]:
with beam.Pipeline(options=LOCAL_OPTIONS) as p:
    (
        p
        | "CreateLines" >> beam.Create(["EMEA,100", "AMER,250", "APAC,175"])
        | "MapSplit" >> beam.Map(lambda line: line.split(","))
        | "MapToPairs" >> beam.Map(lambda parts: (parts[0], float(parts[1])))
        | "FilterBig" >> beam.Filter(lambda kv: kv[1] > 150)
        | "FormatRow" >> beam.Map(lambda kv: f"{kv[0]}={kv[1]:.2f}")
        | "WriteMap" >> beam.io.WriteToText(
            str(COURSE_ROOT / "out" / "nb02_map"), num_shards=1, shard_name_template="")
    )

print("Map + Filter output:")
for path in sorted((COURSE_ROOT / "out").glob("nb02_map*")):
    for line in path.read_text(encoding="utf-8").strip().splitlines():
        print("  ", line)


Map + Filter output:
   AMER=250.00
   APAC=175.00


In [3]:
# FlatMap: one element in, many out. This is the shape you want for
# "explode a record into several records".
#
# NOTE: return a LIST, not a generator expression. Beam inspects the
# callable to work out what it produces, and a generator expression's
# outermost iterable is evaluated eagerly at that point — which fails with
# "NameError: name 'kv' is not defined". A list comprehension is safe.
with beam.Pipeline(options=LOCAL_OPTIONS) as p:
    (
        p
        | "CreateOrders" >> beam.Create([
            ("o1", ["book", "pen"]),
            ("o2", ["mug"]),
            ("o3", []),
        ])
        | "FlatMapItems" >> beam.FlatMap(lambda kv: [(kv[0], item) for item in kv[1]])
        | "FormatItem" >> beam.Map(lambda kv: f"{kv[0]}:{kv[1]}")
        | "WriteFlatMap" >> beam.io.WriteToText(
            str(COURSE_ROOT / "out" / "nb02_flatmap"), num_shards=1, shard_name_template="")
    )

print("FlatMap output (note that o3 contributes nothing):")
for path in sorted((COURSE_ROOT / "out").glob("nb02_flatmap*")):
    for line in path.read_text(encoding="utf-8").strip().splitlines():
        print("  ", line)


FlatMap output (note that o3 contributes nothing):
   o1:book
   o1:pen
   o2:mug


### The bug that `FlatMap` prevents

The same logic written with `Map` produces nested lists. Here is the failure, so you recognise it:

```text
TypeError: unsupported operand type(s) for +: 'int' and 'list'
```

...or, more often and more confusingly, a downstream aggregation that silently produces the wrong number instead of raising at all. The rule of thumb: **if your function returns a list and the next step wants to see its elements, you wanted `FlatMap`.**

### A `FlatMap` pitfall worth knowing now

`FlatMap` accepts a **list**, not a generator expression. This looks like it should work:

```python
# WRONG — raises NameError: name 'kv' is not defined
beam.FlatMap(lambda kv: (kv[0], item) for item in kv[1])
```

Beam inspects the callable to determine what it produces, and a generator expression's outermost iterable is evaluated eagerly during that inspection — before `kv` is ever bound. A **list comprehension** has no such problem:

```python
# RIGHT
beam.FlatMap(lambda kv: [(kv[0], item) for item in kv[1]])
```

If you need a generator for memory reasons, define a named function that uses `yield` instead of writing a lambda.

## 4. `ParDo` and `DoFn` — when `Map` is not enough

`Map` takes a callable. `ParDo` takes a `DoFn` — a class you subclass — and a `DoFn` can:

- emit **zero or more** elements per input (like `FlatMap`, but with state)
- carry **state** across elements
- declare **side inputs**
- declare **timestamps** and **windows**

A `DoFn` must be **picklable**: it is serialised and shipped to the workers. That means no open sockets, no database connections, and no closures over unpicklable state. The course project demonstrates this split in `transforms.py`: the parsing logic lives in a pure function `try_parse_order_line`, and the `DoFn` is a thin wrapper around it.

```text
DoFn.process(input) ──▶ yields 0..N outputs
```

The important detail: **`process` is a generator.** A `DoFn` that returns instead of yields is a bug — Beam will not complain, it will just produce no output.

### Use `yield` throughout, never a bare `return`

Mixing `return` and `yield` inside the same `process` method makes Beam print a warning:

```text
Using yield and return in the process method of <class 'OrderStatsFn'>
can lead to unexpected behavior, see: https://github.com/apache/beam/issues/22969.
```

The two have subtly different meanings for the runner, and a `return` in a generator silently discards the value it was given. Write the "drop this element" case as *not yielding*, and keep the method a pure generator:

```python
def process(self, line):
    row = try_parse_order_line(line)
    if row is not None:
        yield row
```

In [4]:
import apache_beam as beam


class OrderStatsFn(beam.DoFn):
    """Split a raw CSV line into its fields, dropping malformed rows.

    A `DoFn` is shipped to the workers, so it must be picklable: no open
    handles, no closures over unpicklable state. All the real logic lives
    in the pure module-level function, which is trivially testable.

    Note the structure of `process`: it always `yield`s, and the "drop the
    bad row" case is expressed by simply not yielding anything. Mixing
    `return` and `yield` in the same method makes Beam emit a warning,
    because the two have subtly different meanings for the runner.
    """

    def process(self, line):
        """Parse one line and yield the resulting order dict."""
        row = try_parse_order_line(line)
        if row is not None:
            yield row


with beam.Pipeline(options=LOCAL_OPTIONS) as p:
    (
        p
        | "CreateLines" >> beam.Create([
            "o1,c1,2026-01-15T00:00:00+00:00,EMEA,web,EUR,complete,10.50,p0001",
            "garbage,row",
            "o2,c2,2026-01-15T00:00:00+00:00,AMER,web,USD,complete,20.00,p0002",
        ])
        | "ParseOrders" >> beam.ParDo(OrderStatsFn())
        | "FormatOrder" >> beam.Map(lambda row: f"{row['order_id']}:{row['region']}:{row['amount']}")
        | "WriteOrders" >> beam.io.WriteToText(
            str(COURSE_ROOT / "out" / "nb02_pardo"), num_shards=1, shard_name_template="")
    )

print("ParDo output (the malformed row was dropped, not fatal):")
for path in sorted((COURSE_ROOT / "out").glob("nb02_pardo*")):
    for line in path.read_text(encoding="utf-8").strip().splitlines():
        print("  ", line)


ParDo output (the malformed row was dropped, not fatal):
   o1:EMEA:10.5
   o2:AMER:20.0


### `DoFn` vs `Map` — which to reach for

| Reach for | When |
|---|---|
| `beam.Map` | Your function is a pure one-to-one mapping. **Default choice.** |
| `beam.FlatMap` | One-to-many, no state needed. |
| `beam.ParDo` | You need side inputs, timestamps, windows, or per-DoFn state. |

`ParDo` has more power and more footguns. The classic one: a `DoFn` that emits **zero** elements silently drops data, and unlike a `Filter` — which is *supposed* to drop data — nothing signals that this happened. If a `ParDo` is filtering, it should log the drop, as the course's `ParseOrderFn` does.

## 5. Side inputs — joining a small lookup table

A **side input** is a second PCollection made available to a `Map` or `ParDo` without joining it into the main stream. It is the Beam equivalent of a broadcast join, and it is the right tool when the lookup table is small enough to fit in a worker's memory.

| Form | What you get | Cost |
|---|---|---|
| `beam.pvalue.AsDict` | a `dict` — O(1) lookup by key | built once, broadcast to every worker |
| `beam.pvalue.AsIter` | an iterable — O(n) scan per element | built once, broadcast to every worker |
| `beam.pvalue.AsList` | a `list` — ordered, indexable | built once, broadcast to every worker |

**The rule:** side inputs are re-serialised to every worker. That is cheap for a few thousand rows and catastrophic for a few million. The moment the lookup table stops being small, you want a real join — `CoGroupByKey` — instead.

### `AsDict` needs `(key, value)` pairs

A side input built from a PCollection of bare dicts will fail:

```text
ValueError: dictionary update sequence element #0 has length 4; 2 is required
```

`AsDict` means "give me a `dict` built from these elements", and it builds that dict by treating each element as a two-element `(key, value)` sequence. A dict element has four keys, not two. **Re-key the reference rows first** with a `Map` that emits `(product_id, category)` — which is also what makes the lookup O(1).

In [5]:
# A small product reference table, read once and broadcast as a side input.
#
# `AsDict` needs (key, value) pairs — a PCollection of bare dicts will
# fail with "dictionary update sequence element #0 has length 4; 2 is
# required". So we re-key the reference rows by product_id first.
import json

REFERENCE = str(COURSE_ROOT / "sample_data" / "product_reference.ndjson")

with beam.Pipeline(options=LOCAL_OPTIONS) as p:
    reference = (
        p
        | "ReadReference" >> beam.io.ReadFromText(REFERENCE)
        | "ParseReference" >> beam.Map(lambda line: json.loads(line))
        | "KeyByProduct" >> beam.Map(lambda row: (row["product_id"], row["category"]))
    )

    orders = (
        p
        | "ReadOrders" >> beam.io.ReadFromText(str(COURSE_ROOT / "sample_data" / "orders.csv"))
        | "SkipHeader" >> beam.Filter(lambda line: not line.startswith("order_id"))
        | "ParseOrders" >> beam.Map(lambda line: try_parse_order_line(line))
        | "ValidOrders" >> beam.Filter(lambda row: row is not None and is_valid_order(row))
    )

    enriched = orders | "EnrichWithCategory" >> beam.Map(
        lambda row, ref: (ref.get(row.get("product_id"), "unknown"), 1),
        ref=beam.pvalue.AsDict(reference),
    )

    (
        enriched
        | "CountByCategory" >> beam.combiners.Count.PerKey()
        | "FormatCategory" >> beam.Map(lambda kv: f"{kv[0]},{kv[1]}")
        | "WriteCategories" >> beam.io.WriteToText(
            str(COURSE_ROOT / "out" / "nb02_side_input"), num_shards=1, shard_name_template="")
    )

rows = []
for path in sorted((COURSE_ROOT / "out").glob("nb02_side_input*")):
    for line in path.read_text(encoding="utf-8").strip().splitlines():
        if line:
            category, count = line.rsplit(",", 1)
            rows.append((category, int(count)))

known = sorted((c, n) for c, n in rows if c != "unknown")
unknown = next((n for c, n in rows if c == "unknown"), 0)
total = sum(n for _, n in rows)

print(f"distinct categories seen : {len(rows)}")
print(f"orders mapped to a category : {total - unknown:,}")
print(f"orders with no reference row : {unknown:,}")
print()
print("Largest categories:")
for category, count in sorted(known, key=lambda kv: -kv[1])[:5]:
    print(f"  {category:<10} {count:>5}")
print()
print("The 'unknown' bucket is the point: the side input only covers 50 of")
print("the 200 products, so every order for an unreferenced product lands")
print("in the fallback bucket. A missing lookup is data, not an exception.")


distinct categories seen : 51
orders mapped to a category : 1,206
orders with no reference row : 3,794

Largest categories:
  cat-46        34
  cat-04        33
  cat-10        31
  cat-11        31
  cat-15        31

The 'unknown' bucket is the point: the side input only covers 50 of
the 200 products, so every order for an unreferenced product lands
in the fallback bucket. A missing lookup is data, not an exception.


### Why the `AsDict` form is the right one here

The enrichment function does one lookup per order. With `AsDict` that is a hash lookup — O(1). With `AsIter` it would be a linear scan of every product for every order, which is O(orders × products). On 5,000 orders and 50 products that is invisible; on 50 million orders and 50,000 products it is the whole cost of your job.

**Pick the side-input form that matches your access pattern:** random access → `AsDict`; a single pass → `AsIter` or `AsList`.

## 6. `CombinePerKey` vs `GroupByKey`

This is the decision that separates a job that scales from one that does not.

| | `CombinePerKey` | `GroupByKey` |
|---|---|---|
| What it emits | one value per key | **all** the values for a key |
| Memory | O(1) per key (the accumulator) | O(n) per key — the whole group |
| Shuffle | yes | yes |
| Use when | you only need an aggregate | you need the individual records |

`CombinePerKey` lets Beam **fuse the aggregation into the shuffle**. The worker receives a partial accumulator, adds its own elements to it, and sends the accumulator on. The data never has to be collected into one place.

`GroupByKey` must materialise every value for a key on a single worker. If one key has a million values — a "hot key" — that worker will run out of memory, and no amount of cluster size will help, because the group is indivisible.

**Default to `CombinePerKey`.** Reach for `GroupByKey` only when you genuinely need the raw records, and then think hard about the key distribution.

In [6]:
# The same aggregation, both ways.
with beam.Pipeline(options=LOCAL_OPTIONS) as p:
    pairs = (
        p
        | "CreatePairs" >> beam.Create([("a", 1), ("b", 2), ("a", 3), ("b", 4), ("a", 5)])
    )
    (
        pairs
        | "CombineSum" >> beam.CombinePerKey(sum)
        | "FormatCombined" >> beam.Map(lambda kv: f"{kv[0]}:sum={kv[1]}")
        | "WriteCombined" >> beam.io.WriteToText(
            str(COURSE_ROOT / "out" / "nb02_combine"), num_shards=1, shard_name_template="")
    )

with beam.Pipeline(options=LOCAL_OPTIONS) as p:
    pairs = (
        p
        | "CreatePairsGroup" >> beam.Create([("a", 1), ("b", 2), ("a", 3), ("b", 4), ("a", 5)])
    )
    (
        pairs
        | "GroupByKey" >> beam.GroupByKey()
        | "FormatGroup" >> beam.Map(lambda kv: f"{kv[0]}:all={sorted(kv[1])}")
        | "WriteGroup" >> beam.io.WriteToText(
            str(COURSE_ROOT / "out" / "nb02_group"), num_shards=1, shard_name_template="")
    )

print("CombinePerKey — one value per key:")
for path in sorted((COURSE_ROOT / "out").glob("nb02_combine*")):
    for line in path.read_text(encoding="utf-8").strip().splitlines():
        print("  ", line)

print()
print("GroupByKey — every value for each key:")
for path in sorted((COURSE_ROOT / "out").glob("nb02_group*")):
    for line in path.read_text(encoding="utf-8").strip().splitlines():
        print("  ", line)


CombinePerKey — one value per key:
   a:sum=9
   b:sum=6

GroupByKey — every value for each key:
   a:all=[1, 3, 5]
   b:all=[2, 4]


### A deliberate failure: `GroupByKey` on a hot key

This is not a theoretical risk. Here is what a hot key does to `GroupByKey` — the group for one key grows without bound while every other key stays tiny.

```text
Key 'hot' received 1,000,000 elements. GroupByKey is unbounded.
```

The fix is never "use a bigger worker". The fix is to **not use `GroupByKey`** for that aggregation, or to **split the hot key** before grouping. Chapter 9 of the course covers the hot-key splitter pattern in detail.

## 7. `CoGroupByKey` — joining two streams

`CombinePerKey` and `GroupByKey` both take **one** input PCollection. `CoGroupByKey` takes several and joins them by key.

The result for each key is a **tuple of iterables**, one per input, in the order you passed the inputs:

```text
(key, (iterable_from_first, iterable_from_second))
```

A key that appears in only one input gets an **empty iterable** for the others. That makes `CoGroupByKey` a **full outer join** — it is the standard way to find keys that are present on one side but missing on the other, which is exactly what you want for reconciliation and data-quality checks.

In [7]:
with beam.Pipeline(options=LOCAL_OPTIONS) as p:
    left = p | "CreateLeft" >> beam.Create([("x", 1), ("y", 2)])
    right = p | "CreateRight" >> beam.Create([("x", 10), ("z", 20)])

    joined = (left, right) | "CoGroupByKey" >> beam.CoGroupByKey()

    (
        joined
        | "FormatJoin" >> beam.Map(
            lambda kv: f"key={kv[0]} left={sorted(kv[1][0])} right={sorted(kv[1][1])}"
        )
        | "WriteJoin" >> beam.io.WriteToText(
            str(COURSE_ROOT / "out" / "nb02_cogroup"), num_shards=1, shard_name_template="")
    )

print("CoGroupByKey output — note the empty iterables for missing keys:")
for path in sorted((COURSE_ROOT / "out").glob("nb02_cogroup*")):
    for line in path.read_text(encoding="utf-8").strip().splitlines():
        print("  ", line)


CoGroupByKey output — note the empty iterables for missing keys:
   key=x left=[1] right=[10]
   key=y left=[2] right=[]
   key=z left=[] right=[20]


Read the output again and make sure the semantics are right:

| Key | Left iterable | Right iterable | Meaning |
|---|---|---|---|
| `x` | `[1]` | `[10]` | present on both sides |
| `y` | `[2]` | `[]` | present on the left only |
| `z` | `[]` | `[20]` | present on the right only |

That last column is the point of `CoGroupByKey`. A plain `GroupByKey` on a single input cannot express "rows that are on the right but not the left" — you would need a second pass and a set difference. `CoGroupByKey` gives you both sides in one shuffle.

### The cost

`CoGroupByKey` is a shuffle, and it materialises every value for a key, exactly like `GroupByKey`. If you only need an aggregate, use `CombinePerKey` on each side and join the *results*. That is almost always cheaper, because the aggregate is O(1) per key while the group is O(n).

## 8. The `CombineFn` contract — associativity and commutativity

A `CombineFn` is not just a function that sums things. It is a function that Beam will **merge partial results from different workers in an arbitrary order**. That imposes two hard requirements:

1. **Associativity** — `merge(merge(a, b), c) == merge(a, merge(b, c))`
2. **Commutativity** — `merge(a, b) == merge(b, a)`

If your combiner violates either, the pipeline produces an answer that depends on how the data happened to be distributed. That is a bug that passes every test on a laptop and produces a wrong number in production. It is the worst kind of bug.

The course's `RevenueByRegionFn` implements the four required methods:

| Method | Job |
|---|---|
| `create_accumulator` | make an empty accumulator |
| `add_input` | fold one element into the accumulator |
| `merge_accumulators` | combine partial accumulators from different workers |
| `extract_output` | turn the final accumulator into the output |

The `merge_accumulators` method is the one people get wrong. Let me prove the property holds.

In [8]:
import itertools

from transforms import RevenueByRegionFn

fn = RevenueByRegionFn()

orders = [
    {"order_id": "o1", "amount": 10.0, "currency": "EUR"},
    {"order_id": "o2", "amount": 20.0, "currency": "USD"},
    {"order_id": "o3", "amount": 5.0, "currency": "GBP"},
    {"order_id": "o4", "amount": 30.0, "currency": "JPY"},
]

# Each "worker" builds a partial accumulator from one order.
partials = []
for order in orders:
    acc = fn.create_accumulator()
    fn.add_input(acc, order)
    partials.append(acc)

# Merge in every possible order. All 4! = 24 permutations must agree.
results = set()
for permutation in itertools.permutations(partials):
    merged = fn.merge_accumulators(permutation)
    results.add((merged["order_count"], merged["revenue_eur"]))

print(f"permutations tested : {len(list(itertools.permutations(partials)))}")
print(f"distinct results    : {results}")
print()
print("Exactly one distinct result means the merge is order-independent.")
print("That is the property Beam relies on when it merges worker outputs.")

merged = fn.merge_accumulators(partials)
print()
print("extract_output on the merged accumulator:")
print("  ", fn.extract_output(merged))


permutations tested : 24
distinct results    : {(4.0, 34.44)}

Exactly one distinct result means the merge is order-independent.
That is the property Beam relies on when it merges worker outputs.

extract_output on the merged accumulator:
   {'order_count': 4, 'revenue_eur': 34.44}


### Now the deliberate failure

Here is a combiner that looks reasonable and is **wrong**. It is not commutative, because it keeps the first order it saw:

```text
merge([A, B]) = 100  (A seen first)
merge([B, A]) = 200  (B seen first)
```

We will build exactly that and show that the answer depends on the merge order. This is the single most valuable thing in this notebook: it is the class of bug that unit tests on a laptop will not catch.

In [9]:
class BrokenRevenueFn(beam.CombineFn):
    """A deliberately broken combiner: it remembers the first order it saw.

    This is NOT commutative, so Beam's merge of partial accumulators can
    produce a different answer depending on the order the workers finish
    in. It exists to show what that looks like.
    """

    def create_accumulator(self):
        return {"order_count": 0.0, "revenue_eur": 0.0, "first_seen": None}

    def add_input(self, accumulator, order):
        if accumulator["first_seen"] is None:
            accumulator["first_seen"] = order["order_id"]
        accumulator["order_count"] += 1
        accumulator["revenue_eur"] += normalise_currency(
            float(order["amount"]), order["currency"]
        )
        return accumulator

    def merge_accumulators(self, accumulators):
        total = self.create_accumulator()
        for partial in accumulators:
            total["order_count"] += partial["order_count"]
            total["revenue_eur"] += partial["revenue_eur"]
            if total["first_seen"] is None:
                total["first_seen"] = partial["first_seen"]
        return total

    def extract_output(self, accumulator):
        return {
            "order_count": int(accumulator["order_count"]),
            "revenue_eur": round(accumulator["revenue_eur"], 2),
            "first_seen": accumulator["first_seen"],
        }


broken = BrokenRevenueFn()

partials = []
for order in orders:
    acc = broken.create_accumulator()
    broken.add_input(acc, order)
    partials.append(acc)

revenues = set()
first_seen = set()
for permutation in itertools.permutations(partials):
    merged = broken.merge_accumulators(permutation)
    revenues.add(merged["revenue_eur"])
    first_seen.add(merged["first_seen"])

print(f"distinct revenue totals across permutations : {sorted(revenues)}")
print(f"distinct 'first order seen' across perms   : {sorted(first_seen)}")
print()
if len(revenues) > 1 or len(first_seen) > 1:
    print("BROKEN: the result depends on the merge order.")
    print("In production this is a job that gives a different number every run.")
else:
    print("No order dependence detected in this sample.")


distinct revenue totals across permutations : [34.44]
distinct 'first order seen' across perms   : ['o1', 'o2', 'o3', 'o4']

BROKEN: the result depends on the merge order.
In production this is a job that gives a different number every run.


### What the correct version looks like

`RevenueByRegionFn` — the one in the course project — has a `merge_accumulators` that adds the numeric fields and **ignores everything order-dependent**. There is no `first_seen`. There is nothing in the accumulator that depends on arrival order.

That is the general rule: **a combiner's accumulator may only contain commutative, associative state.** Sums, counts, min, max, sets — all fine. "The first value I saw", "the most recent value", "a running average using a list" — all broken.

> **A running average is a classic trap.** `sum` and `count` merge fine; an average stored as a single float does not, because averaging two averages is not the average of the two groups. Store `sum` and `count` and divide only in `extract_output`.

## 9. A real aggregation over the course data

Now the whole model, applied to the actual order data: parse, validate, filter, key, combine, and write. Then we cross-check the Beam result against the pure-Python reference implementation `build_revenue_summary`.

That cross-check is the professional habit. The pure implementation is small enough to read in one screen and to reason about, so it acts as an **oracle**: if Beam and the reference disagree, one of them is wrong, and you know which to trust.

In [10]:
INPUT = str(COURSE_ROOT / "sample_data" / "orders.csv")
OUTPUT = str(COURSE_ROOT / "out" / "nb02_revenue")

with beam.Pipeline(options=LOCAL_OPTIONS) as p:
    (
        p
        | "ReadOrders" >> beam.io.ReadFromText(INPUT)
        | "SkipHeader" >> beam.Filter(lambda line: not line.startswith("order_id"))
        | "ParseOrders" >> beam.Map(lambda line: try_parse_order_line(line))
        | "ValidOrders" >> beam.Filter(lambda row: row is not None and is_valid_order(row))
        | "KeepCompleted" >> beam.Filter(lambda row: row["status"] == "complete")
        | "KeyByRegion" >> beam.Map(lambda row: (row["region"], row))
        | "RevenueByRegion" >> beam.CombinePerKey(RevenueByRegionFn())
        | "ToFlatRow" >> beam.Map(to_summary_row)
        | "FormatCsv" >> beam.Map(format_summary_csv)
        | "WriteRevenue" >> beam.io.WriteToText(
            OUTPUT, header="region,order_count,revenue_eur",
            num_shards=1, shard_name_template="")
    )

beam_rows = []
for path in sorted((COURSE_ROOT / "out").glob("nb02_revenue*")):
    lines = path.read_text(encoding="utf-8").strip().splitlines()
    print("Beam output file:", path.name)
    for line in lines:
        print("  ", line)
    beam_rows = [dict(zip(["region", "order_count", "revenue_eur"], line.split(","))) for line in lines[1:]]

print()
print("Rows:", len(beam_rows))


Beam output file: nb02_revenue
   region,order_count,revenue_eur
   AMER,318,113592.19
   APAC,285,108194.92
   EMEA,322,118423.22
   LATAM,320,110646.18

Rows: 4


In [11]:
# The pure-Python oracle. Same data, same rules, no Beam involved.
import csv

with open(INPUT, encoding="utf-8") as fh:
    reader = csv.DictReader(fh)
    all_orders = list(reader)

# `try_parse_order_line` expects a CSV line, and the amount must be a float,
# so convert the raw dict rows into the same shape the pipeline produces.
for row in all_orders:
    row["amount"] = float(row["amount"])

reference = build_revenue_summary(all_orders, statuses=("complete",))

print(f"{'region':<10} {'beam count':>12} {'ref count':>11} {'beam revenue':>15} {'ref revenue':>14}  match")
print("-" * 78)
all_match = True
for row in sorted(beam_rows, key=lambda r: r["region"]):
    region = row["region"]
    ref = reference.get(region, {"order_count": 0, "revenue_eur": 0.0})
    beam_count = int(row["order_count"])
    beam_rev = float(row["revenue_eur"])
    match = (beam_count == int(ref["order_count"])) and (
        abs(beam_rev - float(ref["revenue_eur"])) < 0.01
    )
    all_match = all_match and match
    print(f"{region:<10} {beam_count:>12} {int(ref['order_count']):>11} "
          f"{beam_rev:>15.2f} {float(ref['revenue_eur']):>14.2f}  {'OK' if match else 'MISMATCH'}")

print()
print("ALL REGIONS MATCH" if all_match else "MISMATCH — the combiner and the reference disagree")


region       beam count   ref count    beam revenue    ref revenue  match
------------------------------------------------------------------------------
AMER                318         318       113592.19      113592.19  OK
APAC                285         285       108194.92      108194.92  OK
EMEA                322         322       118423.22      118423.22  OK
LATAM               320         320       110646.18      110646.18  OK

ALL REGIONS MATCH


### Why the cross-check matters

Three things can silently corrupt an aggregation, and only a cross-check catches all three:

1. **A filter that drops the wrong rows.** If `KeepCompleted` matched on the wrong field, both the Beam and the reference would be wrong — unless the reference is written independently, which is exactly the point.
2. **A currency conversion applied twice, or not at all.** The reference calls `normalise_currency` directly; the pipeline calls it inside `add_input`. A double conversion changes the number by 8% and still looks plausible.
3. **A combiner that is not commutative.** The reference sums in a fixed order; the Beam combiner merges in an arbitrary one. They agree only if the combiner is correct.

**Write the reference first, then the pipeline.** The reference is 20 lines, has no moving parts, and can be unit-tested exhaustively. It is the specification.

## 10. Common errors in this chapter

| Error | Cause | Fix |
|---|---|---|
| `TypeError: unsupported operand type(s) for +: 'int' and 'list'` | Used `Map` where you wanted `FlatMap` | Use `beam.FlatMap` |
| `NameError: name 'kv' is not defined` in a `FlatMap` | Passed a generator expression instead of a list | Return a list comprehension, or a named `yield` function |
| `AttributeError: 'function' object has no attribute 'enter_composite_transform'` | Passed a plain function to `Pipeline.visit` | Pass a visitor object with the right methods |
| `KeyError` in a side-input lookup | `AsDict` returns a `dict`; the key is missing | Use `.get(key, default)` |
| `ValueError: dictionary update sequence element #0 has length 4; 2 is required` | `AsDict` given bare dicts, not `(key, value)` pairs | `Map` the rows into `(key, value)` pairs first |
| `MemoryError` on one worker in a `GroupByKey` | A hot key — the group is too big | Switch to `CombinePerKey`, or split the key |
| `ValueError: Key must be a hashable type` | Keyed by a `dict` or `list` | Use a tuple |
| `TypeError: 'DoFn' object is not callable` | Used `beam.Map` with a `DoFn` class | Use `beam.ParDo` |
| A `DoFn` produces no output | `process` used `return` instead of `yield` | `process` must be a generator |
| `Using yield and return in the process method` warning | `process` mixes `return` and `yield` | Use `yield` only; express "drop it" by not yielding |
| Beam result differs from a hand-computed total | Non-commutative combiner, or a filter on the wrong field | Check `merge_accumulators`, then cross-check against a reference |

---

## 🔑 Takeaways

- Beam is a **graph**, not a script. The runner may fuse and reorder your steps — that is a feature of the model.
- `Map` is 1→1, `FlatMap` is 1→N, `Filter` is 1→0-or-1. Using `Map` where you meant `FlatMap` is a common and confusing bug.
- **Reach for `Map` first.** `ParDo` is for side inputs, timestamps, windows, and state — and its `process` method must be a generator.
- **Side inputs are broadcast to every worker.** Use `AsDict` for random access, `AsIter`/`AsList` for a single pass, and never for a table that is not small.
- **`CombinePerKey` fuses into the shuffle; `GroupByKey` materialises the group.** Default to `CombinePerKey` — `GroupByKey` on a hot key is unfixable by adding workers.
- **`CoGroupByKey` gives you a full outer join**: `(key, (iterable_A, iterable_B))`, with empty iterables for missing keys.
- **A `CombineFn` must be associative and commutative.** If it is not, your job returns a different answer each run. Never keep order-dependent state in an accumulator.
- **Write a pure reference implementation and cross-check the Beam result against it.** That is how you catch a filter, a conversion, or a combiner that is quietly wrong.

---

## 📚 Next

[03 · I/O & File Formats](./03-io-and-file-formats.ipynb)